### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="aps_failure",
    dataset_year="2016",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5V60Q",
    download_description="""
    mkdir -p local-data-warehouse/aps_failure 
    wget -P local-data-warehouse/aps_failure/ https://archive.ics.uci.edu/static/public/414/ida2016challenge.zip 
    unzip local-data-warehouse/aps_failure/ida2016challenge.zip -d local-data-warehouse/aps_failure/ 
    rm local-data-warehouse/aps_failure/ida2016challenge.zip
""",
    # References
    academic_reference_bibtex="""@misc{ida2016challenge,
  author       = {{IDA2016Challenge}},
  title        = {IDA2016Challenge [Dataset]},
  year         = {2016},
  howpublished = {\\url{https://doi.org/10.24432/C5V60Q}},
  note         = {UCI Machine Learning Repository},
}
""",
    academic_reference_bibtex_key="ida2016challenge",
    license="CC BY 4.0",
    data_tags=["ForcedIIDFromTemporal", "Anonymized"],
    curation_comments="""
  - We combined the original training and testing data into a single dataset.
  - We renamed the target feature to "AirPressureSystemFailure".
  - We converted "na" strings to real NaN/missing values, making the data numeric.
  - Anomaly: we cannot determine the data types of the features.
  - Anomaly: some features are bins of histograms (see original data description).
  - Anomaly: the original task used a cost matrix for evaluation.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="AirPressureSystemFailure",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="AirPressureSystemFailure",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.concat(
    [
        pd.read_csv(f"{dataset_mold.path}/to_uci/aps_failure_training_set.csv", skiprows=20, na_values="na"),
        pd.read_csv(f"{dataset_mold.path}/to_uci/aps_failure_test_set.csv", skiprows=20, na_values="na"),
    ]
)

df.rename(columns={"class": "AirPressureSystemFailure"}, inplace=True)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Loaded data shape:", df.shape)

Loaded data shape: (76000, 171)


In [3]:
# Use if needed to get see all cols of pandas dataframes
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)
df.head()

,AirPressureSystemFailure,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,ag_003,ag_004,ag_005,ag_006,ag_007,ag_008,ag_009,ah_000,ai_000,aj_000,ak_000,al_000,am_0,an_000,ao_000,ap_000,aq_000,ar_000,as_000,at_000,au_000,av_000,ax_000,ay_000,ay_001,ay_002,ay_003,ay_004,ay_005,ay_006,ay_007,ay_008,ay_009,az_000,az_001,az_002,az_003,az_004,az_005,az_006,az_007,az_008,az_009,ba_000,ba_001,ba_002,ba_003,ba_004,ba_005,ba_006,ba_007,ba_008,ba_009,bb_000,bc_000,bd_000,be_000,bf_000,bg_000,bh_000,bi_000,bj_000,bk_000,bl_000,bm_000,bn_000,bo_000,bp_000,bq_000,br_000,bs_000,bt_000,bu_000,bv_000,bx_000,by_000,bz_000,ca_000,cb_000,cc_000,cd_000,ce_000,cf_000,cg_000,ch_000,ci_000,cj_000,ck_000,cl_000,cm_000,cn_000,cn_001,cn_002,cn_003,cn_004,cn_005,cn_006,cn_007,cn_008,cn_009,co_000,cp_000,cq_000,cr_000,cs_000,cs_001,cs_002,cs_003,cs_004,cs_005,cs_006,cs_007,cs_008,cs_009,ct_000,cu_000,cv_000,cx_000,cy_000,cz_000,da_000,db_000,dc_000,dd_000,de_000,df_000,dg_000,dh_000,di_000,dj_000,dk_000,dl_000,dm_000,dn_000,do_000,dp_000,dq_000,dr_000,ds_000,dt_000,du_000,dv_000,dx_000,dy_000,dz_000,ea_000,eb_000,ec_00,ed_000,ee_000,ee_001,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,4090,0.0,268.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79820.0,0.0,0.0,0.0,30902.0,50622.0,242674.0,244736.0,54112.0,13910.0,0.0,0.0,0.0,0.0,132.0,208.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,327848.0,138.0,114.0,284.0,4.0,79820.0,1820.0,22336.0,31272.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52820.0,4089.02,327848.0,327848.0,296.0,0.0,92.0,31980.0,326140.0,0.0,1209600.0,4826.0,0.0,0.0,0.0,200013.12,4826.88,79081.92,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,327848.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,590.0,398.0,10430.0,6734.0,0.0,0.0,0.0,0.0,0.0,0.0,2232.0,18.0,6.0,0.0,0.0,4288.0,638.0,987500.0,98836.0,0.0,0.0,0.0,0.0,3089100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,neg,12,0.0,8.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,2328.0,1006.0,1700.0,0.0,0.0,0.0,622.0,0.0,0.0,0.0,442.0,792.0,11258.0,10292.0,18590.0,366.0,0.0,0.0,0.0,0.0,14.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,430.0,4604.0,0.0,40.0,362.0,752.0,44.0,12.0,140.0,3684.0,0.0,0.0,0.0,3532.0,336.0,80.0,80.0,266.0,56.0,454.0,56.0,18.0,156.0,30308.0,0.0,6.0,0.0,0.0,622.0,110.0,16982.0,1526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18400.0,12.59,30308.0,30308.0,33030.0,27.0,18.0,2976.0,30100.0,5066.0,1209600.0,270.0,2.0,2.0,0.0,1973.76,0.00,1933.44,0.0,6.0,0.0,0.0,2170.0,346.0,1586.0,354.0,578.0,0.0,0.0,0.0,0.0,0.0,30308.0,0.0,1198.0,8.0,18.0,2.0,30.0,624.0,2356.0,772.0,26.0,0.0,4.0,20.0,2108.0,74.0,0.0,0.0,0.0,0.0,2170.0,32.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,158.0,0.0,0.0,0.0,0.0,20.0,16.0,7260.0,728.0,0.0,6.0,0.0,0.0,0.0,0.58,30.0,3258.0,1532.0,30.0,8.0,38.0,32.0,60.0,76.0,0.0,0.0,0.0,0.0
2,neg,378,NaN,32.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,148.0,3212.0,28760.0,1020.0,0.0,0.0,16812.0,0.0,0.0,0.0,0.0,0.0,28814.0,21958.0,37052.0,2696.0,0.0,0.0,0.0,0.0,18.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,29092.0,2296.0,1752.0,0.0,1786.0,30.0,148.0,144.0,488.0,30266.0,278.0,0.0,0.0,0.0,22570.0,7592.0,2202.0,270.0,78.0,146.0,278.0,4.0,0.0,0.0,65890.0,2.0,58.0,22.0,0.0,16812.0,494.0,30462.0,6552.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36740.0,377.91,65890.0,65890.0,67674.0,98.0,0.0,6298.0,65580.0,33140.0,1209600.0,944.0,2.0,0.0,0.0,22626.24,0.00,8659.20,NaN,NaN,0.0,0.0,0.0,1266.0,18886.0,12068.0,800.0,100.0,20.0,0.0,0.0,4.0,65890.0,NaN,1880.0,24.0,504.0,1644.0,536.0,14222.0,12174.0,2072.0,84.0,0.0,62.0,70.0,19596.0,12412.0,42.0,42.0,0.0,0.0,20494.0,94.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,448.0,4.0,2.0,0.0,0.0,528.0,122.0,1100.0,110.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,11074.0,5894.0,1324.0,346.0,386.0,14064.0,22.0,30.0,0.0,0.0,0.0,0.0
3,neg,10616,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,6882.0,120030.0,420100.0,212040.0,10278.0,0.0,0.0,301262.0,0.0,0.0,0.0,5332.0,6890.0,745736.0,722518.0,

## Data Checks

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 76,000
Columns: 171
Use sampling: False (sample size: 76,000)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['bx_000', 'bb_000', 'bu_000', 'cq_000', 'bv_000', 'an_000', 'ao_000', 'ci_000', 'bt_000', 'ck_000']
Rows remaining as candidates after top-10 filter: 107 (of 76,000)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,AirPressureSystemFailure,aa_000,ab_000,ac_000,ad_000,ae_000,af_000,ag_000,ag_001,ag_002,ag_003,ag_004,ag_005,ag_006,ag_007,ag_008,ag_009,ah_000,ai_000,aj_000,ak_000,al_000,am_0,an_000,ao_000,ap_000,aq_000,ar_000,as_000,at_000,au_000,av_000,ax_000,ay_000,ay_001,ay_002,ay_003,ay_004,ay_005,ay_006,ay_007,ay_008,ay_009,az_000,az_001,az_002,az_003,az_004,az_005,az_006,az_007,az_008,az_009,ba_000,ba_001,ba_002,ba_003,ba_004,ba_005,ba_006,ba_007,ba_008,ba_009,bb_000,bc_000,bd_000,be_000,bf_000,bg_000,bh_000,bi_000,bj_000,bk_000,bl_000,bm_000,bn_000,bo_000,bp_000,bq_000,br_000,bs_000,bt_000,bu_000,bv_000,bx_000,by_000,bz_000,ca_000,cb_000,cc_000,cd_000,ce_000,cf_000,cg_000,ch_000,ci_000,cj_000,ck_000,cl_000,cm_000,cn_000,cn_001,cn_002,cn_003,cn_004,cn_005,cn_006,cn_007,cn_008,cn_009,co_000,cp_000,cq_000,cr_000,cs_000,cs_001,cs_002,cs_003,cs_004,cs_005,cs_006,cs_007,cs_008,cs_009,ct_000,cu_000,cv_000,cx_000,cy_000,cz_000,da_000,db_000,dc_000,dd_000,de_000,df_000,dg_000,dh_000,di_000,dj_000,dk_000,dl_000,dm_000,dn_000,do_000,dp_000,dq_000,dr_000,ds_000,dt_000,du_000,dv_000,dx_000,dy_000,dz_000,ea_000,eb_000,ec_00,ed_000,ee_000,ee_001,ee_002,ee_003,ee_004,ee_005,ee_006,ee_007,ee_008,ee_009,ef_000,eg_000
0,neg,4090,0.0,268.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,79820.0,0.0,0.0,0.0,30902.0,50622.0,242674.0,244736.0,54112.0,13910.0,0.0,0.0,0.0,0.0,132.0,208.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,327848.0,138.0,114.0,284.0,4.0,79820.0,1820.0,22336.0,31272.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,52820.0,4089.02,327848.0,327848.0,296.0,0.0,92.0,31980.0,326140.0,0.0,1209600.0,4826.0,0.0,0.0,0.0,200013.12,4826.88,79081.92,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,327848.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,590.0,398.0,10430.0,6734.0,0.0,0.0,0.0,0.0,0.0,0.0,2232.0,18.0,6.0,0.0,0.0,4288.0,638.0,987500.0,98836.0,0.0,0.0,0.0,0.0,3089100.0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,neg,12,0.0,8.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,2328.0,1006.0,1700.0,0.0,0.0,0.0,622.0,0.0,0.0,0.0,442.0,792.0,11258.0,10292.0,18590.0,366.0,0.0,0.0,0.0,0.0,14.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,430.0,4604.0,0.0,40.0,362.0,752.0,44.0,12.0,140.0,3684.0,0.0,0.0,0.0,3532.0,336.0,80.0,80.0,266.0,56.0,454.0,56.0,18.0,156.0,30308.0,0.0,6.0,0.0,0.0,622.0,110.0,16982.0,1526.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18400.0,12.59,30308.0,30308.0,33030.0,27.0,18.0,2976.0,30100.0,5066.0,1209600.0,270.0,2.0,2.0,0.0,1973.76,0.00,1933.44,0.0,6.0,0.0,0.0,2170.0,346.0,1586.0,354.0,578.0,0.0,0.0,0.0,0.0,0.0,30308.0,0.0,1198.0,8.0,18.0,2.0,30.0,624.0,2356.0,772.0,26.0,0.0,4.0,20.0,2108.0,74.0,0.0,0.0,0.0,0.0,2170.0,32.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,158.0,0.0,0.0,0.0,0.0,20.0,16.0,7260.0,728.0,0.0,6.0,0.0,0.0,0.0,0.58,30.0,3258.0,1532.0,30.0,8.0,38.0,32.0,60.0,76.0,0.0,0.0,0.0,0.0
2,neg,378,NaN,32.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,148.0,3212.0,28760.0,1020.0,0.0,0.0,16812.0,0.0,0.0,0.0,0.0,0.0,28814.0,21958.0,37052.0,2696.0,0.0,0.0,0.0,0.0,18.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,29092.0,2296.0,1752.0,0.0,1786.0,30.0,148.0,144.0,488.0,30266.0,278.0,0.0,0.0,0.0,22570.0,7592.0,2202.0,270.0,78.0,146.0,278.0,4.0,0.0,0.0,65890.0,2.0,58.0,22.0,0.0,16812.0,494.0,30462.0,6552.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36740.0,377.91,65890.0,65890.0,67674.0,98.0,0.0,6298.0,65580.0,33140.0,1209600.0,944.0,2.0,0.0,0.0,22626.24,0.00,8659.20,NaN,NaN,0.0,0.0,0.0,1266.0,18886.0,12068.0,800.0,100.0,20.0,0.0,0.0,4.0,65890.0,NaN,1880.0,24.0,504.0,1644.0,536.0,14222.0,12174.0,2072.0,84.0,0.0,62.0,70.0,19596.0,12412.0,42.0,42.0,0.0,0.0,20494.0,94.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,448.0,4.0,2.0,0.0,0.0,528.0,122.0,1100.0,110.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,11074.0,5894.0,1324.0,346.0,386.0,14064.0,22.0,30.0,0.0,0.0,0.0,0.0
3,neg,10616,NaN,0.0,NaN,0.0,0.0,0.0,0.0,0.0,6882.0,120030.0,420100.0,212040.0,10278.0,0.0,0.0,301262.0,0.0,0.0,0.0,5332.0,6890.0,745736.0,722518.0,

In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,br_000,float64,62393.0,82.10,4607.0,"1310700.0, 0.0, 229360.0, 205800.0, 267280.0, 218560.0, 250840.0, 292360.0, 181280.0, 288400.0"
1,bq_000,float64,61703.0,81.19,5116.0,"1310700.0, 0.0, 220000.0, 165940.0, 349100.0, 308480.0, 267400.0, 136780.0, 214300.0, 153480.0"
2,bp_000,float64,60461.0,79.55,5863.0,"1310700.0, 0.0, 168060.0, 289660.0, 161080.0, 241360.0, 235700.0, 216480.0, 196220.0, 204920.0"
3,bo_000,float64,58709.0,77.25,6828.0,"1310700.0, 0.0, 221400.0, 235860.0, 377320.0, 209740.0, 133900.0, 208740.0, 272300.0, 403280.0"
4,ab_000,float64,58692.0,77.23,30.0,"0.0, 2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 14.0, 16.0, 24.0"
5,cr_000,float64,58692.0,77.23,86.0,"0.0, 44.0, 3750.0, 1234.0, 4.0, 1416.0, 54.0, 52.0, 450.0, 2632.0"
6,bn_000,float64,55722.0,73.32,8216.0,"1310700.0, 0.0, 182360.0, 254940.0, 222060.0, 300080.0, 162980.0, 184040.0, 198080.0, 261620.0"
7,bm_000,float64,50095.0,65.91,10254.0,"1310700.0, 0.0, 193240.0, 198100.0, 237100.0, 259560.0, 212200.0, 228140.0, 159060.0, 178380.0"
8,bl_000,float64,34503.0,45.40,13085.0,"1310700.0, 0.0, 156260.0, 174560.0, 160580.0, 213400.0, 172900.0, 179340.0, 181280.0, 169300.0"
9,bk_000,float64,29128.0,38.33,14029.0,"1310700.0, 0.0, 161880.0, 179220.0, 212520.0, 167800.0, 186480.0, 145460.0, 175300.0, 171740.0"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
aa_000,76000.0,6.115976e+04,2.647366e+05,0.0,4.294967e+07
ab_000,17308.0,7.255604e-01,3.311913e+00,0.0,2.040000e+02
ac_000,71739.0,3.564398e+08,7.952530e+08,0.0,2.130707e+09
ad_000,57158.0,1.506300e+05,3.590593e+07,0.0,8.584298e+09
ae_000,72810.0,6.736959e+00,1.534465e+02,0.0,2.105000e+04
af_000,72810.0,1.083631e+01,2.015949e+02,0.0,2.007000e+04
ag_000,75140.0,2.004306e+02,1.843612e+04,0.0,3.376892e+06
ag_001,75140.0,1.204761e+03,5.099091e+04,0.0,1.047252e+07
ag_002,75140.0,9.697328e+03,1.718996e+05,0.0,1.914916e+07
ag_003,75140.0,9.364902e+04,8.244157e+05,0.0,7.305747e+07


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column                   rank                    
AirPressureSystemFailure 1      neg  74625  98.19
                         2      pos   1375   1.81

In [9]:
# Target Distribution
target_df

,count,pct
AirPressureSystemFailure,,
neg,74625,98.19
pos,1375,1.81


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019ce752-57f8-7535-90f8-ce435bb1027f
bbb6acbb2087cbd107d542322409c797b5ea869fb5200c067d6f60ce306fee8c
